# Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

### Objective:
- Combine all previous concepts (`CSVReader`, Strategy Pattern, Decorators, Factory Pattern) into one unified **cleaning pipeline**.
- Use the **Template Method Pattern** to define a consistent, reproducible workflow.

### Concept:
The **Template Method Pattern** defines the skeleton of an algorithm in an abstract base class's method (here: `run()`), delegating the specific implementations of individual steps (`load`, `clean`, `transform`, `save`) to concrete subclasses.
This guarantees that all data pipelines follow the exact same lifecycle without duplicating workflow orchestration logic.

## 1. Defining the Abstract DataPipeline (Template Method)

The `DataPipeline` class outlines the four sequential steps:
1. `load()`
2. `clean()`
3. `transform()`
4. `save()`

The `run()` method is our **Template Method** calling each step in order.

In [1]:
import time
import functools
import pandas as pd
from abc import ABC, abstractmethod
from typing import List

# --- Decorators from Exercise 3 ---
def log_action(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"\n>>> [STEP] Executing: {func.__name__}()")
        res = func(*args, **kwargs)
        print(f"<<< [DONE] Finished: {func.__name__}()")
        return res
    return wrapper

def log_time(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.time()
        res = func(*args, **kwargs)
        dt = time.time() - t0
        print(f"    [TIME] {func.__name__}() took {dt:.4f}s")
        return res
    return wrapper

# --- Abstract Pipeline with Template Method ---
class DataPipeline(ABC):
    """
    Abstract Base Class defining the Template Method for Data Pipelines.
    """
    def __init__(self):
        self.data = None

    def run(self):
        """
        The Template Method: Defines the rigid sequence of steps.
        Subclasses implement the individual step details.
        """
        print("=" * 55)
        print("  STARTING DATA CLEANING & TRANSFORMATION PIPELINE")
        print("=" * 55)
        
        self.load()
        self.clean()
        self.transform()
        self.save()
        
        print("=" * 55)
        print("  PIPELINE FINISHED SUCCESSFULLY!")
        print("=" * 55)
        return self.data

    @abstractmethod
    def load(self):
        """Step 1: Load data from source."""
        pass

    @abstractmethod
    def clean(self):
        """Step 2: Clean missing values / outliers."""
        pass

    @abstractmethod
    def transform(self):
        """Step 3: Apply transformations & feature scaling."""
        pass

    @abstractmethod
    def save(self):
        """Step 4: Save cleaned data to destination."""
        pass

## 2. Incorporating Previous Classes

We bring in our previously developed components:
- `CSVReader` (from Exercise 1 & 3)
- `MissingValueStrategy` & `FillMean` (from Exercise 2)
- `TransformFactory` & transformations (from Exercise 4)

In [2]:
# --- Component: CSVReader ---
class CSVReader:
    def __init__(self, file_path: str):
        self.file_path = file_path
        
    def read(self) -> pd.DataFrame:
        print(f"  Reading CSV from '{self.file_path}'...")
        return pd.read_csv(self.file_path)

# --- Component: Strategy Pattern for Cleaning ---
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

class FillMeanAndMode(MissingValueStrategy):
    """Combines numeric mean imputation with categorical mode imputation."""
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        cleaned = df.copy()
        # Fill numeric with mean
        for col in cleaned.select_dtypes(include=['number']).columns:
            if cleaned[col].isnull().sum() > 0:
                mean_val = cleaned[col].mean()
                cleaned[col] = cleaned[col].fillna(mean_val)
                print(f"  - Imputed '{col}' with mean = {mean_val:.1f}")
        # Fill categorical with mode
        for col in cleaned.select_dtypes(include=['object', 'string']).columns:
            if cleaned[col].isnull().sum() > 0:
                mode_val = cleaned[col].mode().iloc[0]
                cleaned[col] = cleaned[col].fillna(mode_val)
                print(f"  - Imputed '{col}' with mode = '{mode_val}'")
        return cleaned

class DataCleaner:
    def __init__(self, strategy: MissingValueStrategy):
        self.strategy = strategy
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.strategy.handle(df)

# --- Component: Factory Pattern for Transformations ---
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

class NormalizeColumns(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        res = df.copy()
        cols = [c for c in res.select_dtypes(include=['number']).columns if c != 'id']
        for col in cols:
            mn, mx = res[col].min(), res[col].max()
            if mx > mn:
                res[col] = (res[col] - mn) / (mx - mn)
        print("  - Applied min-max normalization to numerical columns.")
        return res

class RemoveDuplicates(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        before = len(df)
        res = df.drop_duplicates(subset=[c for c in df.columns if c != 'id']).copy()
        print(f"  - Removed {before - len(res)} duplicate records.")
        return res

class StandardizeText(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        res = df.copy()
        for col in res.select_dtypes(include=['object', 'string']).columns:
            res[col] = res[col].astype(str).str.strip().str.lower()
        print("  - Standardized text strings to lower-case.")
        return res

class TransformFactory:
    @staticmethod
    def get_transform(name: str) -> DataTransform:
        k = name.lower().strip()
        if k in ["normalize", "normalize_columns"]:
            return NormalizeColumns()
        elif k in ["remove_duplicates", "deduplicate"]:
            return RemoveDuplicates()
        elif k in ["standardize_text", "clean_text"]:
            return StandardizeText()
        raise ValueError(f"Unknown transform: {name}")

## 3. Implementing the Concrete CSVDataPipeline

`CSVDataPipeline` implements the abstract methods of `DataPipeline` by composing the reader, cleaner, and factory.

In [3]:
class CSVDataPipeline(DataPipeline):
    """
    Concrete pipeline that loads a CSV, cleans missing values,
    applies a series of transformations, and saves the output.
    """
    def __init__(self, 
                 input_path: str, 
                 output_path: str, 
                 clean_strategy: MissingValueStrategy,
                 transform_steps: List[str]):
        super().__init__()
        self.input_path = input_path
        self.output_path = output_path
        self.cleaner = DataCleaner(clean_strategy)
        self.transform_steps = transform_steps

    @log_action
    @log_time
    def load(self):
        reader = CSVReader(self.input_path)
        self.data = reader.read()
        print(f"  Loaded dataset with shape: {self.data.shape}")

    @log_action
    @log_time
    def clean(self):
        self.data = self.cleaner.clean(self.data)
        print(f"  Cleaned dataset. Missing values count: {self.data.isnull().sum().sum()}")

    @log_action
    @log_time
    def transform(self):
        for step_name in self.transform_steps:
            transformer = TransformFactory.get_transform(step_name)
            self.data = transformer.apply(self.data)

    @log_action
    @log_time
    def save(self):
        self.data.to_csv(self.output_path, index=False)
        print(f"  Saved final cleaned data to '{self.output_path}'")

## 4. Running the Full Pipeline

In [4]:
# Initialize and execute the CSVDataPipeline
pipeline = CSVDataPipeline(
    input_path="sample_data.csv",
    output_path="cleaned_data.csv",
    clean_strategy=FillMeanAndMode(),
    transform_steps=["remove_duplicates", "standardize_text", "normalize"]
)

# Run the complete pipeline via the Template Method!
final_df = pipeline.run()

  STARTING DATA CLEANING & TRANSFORMATION PIPELINE

>>> [STEP] Executing: load()
  Reading CSV from 'sample_data.csv'...
  Loaded dataset with shape: (8, 5)
    [TIME] load() took 0.0050s
<<< [DONE] Finished: load()

>>> [STEP] Executing: clean()
  - Imputed 'age' with mean = 30.5
  - Imputed 'salary' with mean = 62142.9
  - Imputed 'city' with mode = 'New York'
  Cleaned dataset. Missing values count: 0
    [TIME] clean() took 0.0052s
<<< [DONE] Finished: clean()

>>> [STEP] Executing: transform()
  - Removed 1 duplicate records.
  - Standardized text strings to lower-case.
  - Applied min-max normalization to numerical columns.
    [TIME] transform() took 0.0036s
<<< [DONE] Finished: transform()

>>> [STEP] Executing: save()
  Saved final cleaned data to 'cleaned_data.csv'
    [TIME] save() took 0.0015s
<<< [DONE] Finished: save()
  PIPELINE FINISHED SUCCESSFULLY!


In [5]:
# Verify the final cleaned and transformed DataFrame
print("Final Cleaned & Transformed Data:")
display(final_df)

Final Cleaned & Transformed Data:


,id,name,age,salary,city
0,1,alice,0.000000,0.000000,new york
1,2,bob,0.333333,0.303571,paris
2,3,charlie,0.366667,0.500000,london
3,4,david,1.000000,1.000000,tokyo
5,6,eve,0.666667,0.375000,paris
6,7,frank,0.200000,0.050000,london
7,8,grace,0.366667,0.200000,new york


In [6]:
# Inspect saved file on disk
saved_check = pd.read_csv("cleaned_data.csv")
print("Verification of saved CSV file on disk:")
display(saved_check)

Verification of saved CSV file on disk:


,id,name,age,salary,city
0,1,alice,0.000000,0.000000,new york
1,2,bob,0.333333,0.303571,paris
2,3,charlie,0.366667,0.500000,london
3,4,david,1.000000,1.000000,tokyo
4,6,eve,0.666667,0.375000,paris
5,7,frank,0.200000,0.050000,london
6,8,grace,0.366667,0.200000,new york


### Summary:
- Implemented the **Template Method Pattern** via `DataPipeline` and `run()`.
- Successfully combined OOP Encapsulation (`CSVReader`), Strategy Pattern (`DataCleaner`), Decorators (`@log_action`, `@log_time`), and Factory Pattern (`TransformFactory`).
- The entire process is modular, robust, and clean.